# SNI v2 multiresolution failure audit

Audit validation-only tanpa training dan tanpa GPU. Membandingkan prediction S2G GAP dengan S2MR multiresolusi pada tingkat crop, source-group/class, kelas, dan domain Adrian/Faruq. Test tetap terkunci.

In [ ]:
# 1/3 - Setup repository, Drive, dan artefak
from google.colab import drive, userdata
from pathlib import Path
import json, os, subprocess, sys

drive.mount('/content/drive')
REPO = Path('/content/coffee-bean-classification')
REF = 'agent/sni-instance-crops'
if not (REPO / '.git').is_dir():
    subprocess.run(['git', 'clone', '--branch', REF, '--single-branch', 'https://github.com/ediprin/coffee-bean-classification.git', str(REPO)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO), 'fetch', 'origin', REF], check=True)
    subprocess.run(['git', '-C', str(REPO), 'checkout', REF], check=True)
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only', 'origin', REF], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(REPO)], check=True)

DRIVE_DATA = Path('/content/drive/MyDrive/coffee-sni-instance-crop-v1')
MANIFEST_ROOT = DRIVE_DATA / 'classification-v2'
RESULT_ROOT = Path('/content/drive/MyDrive/sni-v2-multiresolution-v1')
AUDIT_ROOT = RESULT_ROOT / 'failure_audit_seed42'
HF_REPO = 'ediprin/coffee-backbone-checkpoints'
HF_NAMESPACE = 'sni-classification-v2-multiresolution'
assert (MANIFEST_ROOT / 'audit.json').is_file(), f'Manifest v2 tidak ditemukan: {MANIFEST_ROOT}'
print('MANIFEST:', MANIFEST_ROOT)
print('RESULT  :', RESULT_ROOT)

In [ ]:
# 2/3 - Pastikan predictions tersedia; pulihkan dari HF bila perlu
prediction_paths = {
    code: RESULT_ROOT / 'val_reports' / f'{code}_seed42' / 'predictions.csv'
    for code in ('S2G', 'S2MR')
}
if not all(path.is_file() for path in prediction_paths.values()):
    token = userdata.get('HF_TOKEN')
    assert token, 'Tambahkan secret HF_TOKEN agar report dapat dipulihkan.'
    os.environ['HF_TOKEN'] = token
    from bilinear_lmmd.core.artifact_store import restore_artifacts
    for code, prediction_path in prediction_paths.items():
        report_dir = prediction_path.parent
        restored = restore_artifacts(
            HF_REPO,
            f'{HF_NAMESPACE}/val_reports/{code}_seed42',
            report_dir,
            filenames=('metrics.json', 'confusion_matrix.csv', 'predictions.csv'),
            overwrite=False,
        )
        print(f'RESTORE {code}: {len(restored)} file')
for code, path in prediction_paths.items():
    assert path.is_file(), f'Predictions {code} belum tersedia: {path}'
    print(code, path)

In [ ]:
# 3/3 - Jalankan audit dan tampilkan tabel utama
command = [
    sys.executable, '-u', '-m', 'bilinear_lmmd.analysis.sni_v2_failure',
    '--manifest-root', str(MANIFEST_ROOT),
    '--baseline-predictions', str(prediction_paths['S2G']),
    '--candidate-predictions', str(prediction_paths['S2MR']),
    '--output-dir', str(AUDIT_ROOT),
]
print('MENJALANKAN:', ' '.join(command), flush=True)
subprocess.run(command, cwd=REPO, check=True)

import pandas as pd
report = json.loads((AUDIT_ROOT / 'sni_v2_failure_audit.json').read_text())
print('\n=== KELAS PALING DIRUSAK S2MR ===')
display(pd.read_csv(AUDIT_ROOT / 'class_diagnostics.csv').sort_values('delta_f1').head(10))
print('\n=== DOMAIN ===')
display(pd.read_csv(AUDIT_ROOT / 'domain_metrics.csv'))
print('\n=== CONFUSION TERBESAR ===')
display(pd.read_csv(AUDIT_ROOT / 'confusion_pairs.csv').head(20))
print('TEST DIBUKA:', report['test_opened'])
print('SAVED:', AUDIT_ROOT)